In [1]:
import triton
import torch
from torch import nn
from typing import Optional, Tuple
import torch.nn.functional as F
import triton.language as tl
import time

torch.set_default_tensor_type(torch.cuda.HalfTensor)

/home/vishwa/.local/lib/python3.10/site-packages/torch/__init__.py:747: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:431.)
  _C._set_default_tensor_type(t)


In [2]:
class FeedForward(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        multiple_of: int,
        ffn_dim_multiplier: Optional[float],
    ):
        super().__init__()
        hidden_dim = int(2 * hidden_dim / 3)
        # custom dim factor multiplier
        if ffn_dim_multiplier is not None:
            hidden_dim = int(ffn_dim_multiplier * hidden_dim)
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)
        self.w1 = torch.nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = torch.nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = torch.nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.feed_forward = FeedForward(dim = 4096, hidden_dim = 16384, multiple_of = 1024, ffn_dim_multiplier = 1.3)
        self.ffn_norm = RMSNorm(dim=4096, eps=1e-5)

    def forward(
        self,
        x: torch.Tensor,
        # start_pos: int,
        # freqs_cis: torch.Tensor,
        # mask: Optional[torch.Tensor],
    ):
        out = self.feed_forward(self.ffn_norm(x))
        return out
        
ff = TransformerBlock().cuda()

In [10]:
@triton.jit
def ff_llama(
    a_ptr, w1_ptr, w3_ptr, out_ptr, rms_w_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_w1k, stride_w1n,
    stride_w3k, stride_w3n,
    stride_outm, stride_outn,
    stride_rms_w,
    EPS: tl.constexpr,
    BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
):
    """
    w1 and w3 are weights (linear layers)
    F.silu(w1(x)) * w3(x)
    """
    pid = tl.program_id(axis=0)
    pid_m = pid // tl.cdiv(N, BLOCK_SIZE_N)
    pid_n = pid % tl.cdiv(N, BLOCK_SIZE_N)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    w1_ptrs = w1_ptr + (offs_k[:, None] * stride_w1k + offs_bn[None, :] * stride_w1n)
    w3_ptrs = w3_ptr + (offs_k[:, None] * stride_w3k + offs_bn[None, :] * stride_w3n)
    acc1 = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    acc2 = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    rms_w_ptrs = rms_w_ptr + tl.arange(0, BLOCK_SIZE_K)[None, :] * stride_rms_w
    a_sum = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_K), dtype=tl.float32)
    for _ in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs)
        # a_sum += tl.pow(a.to(tl.float32), 2)
        a_sum += a * a
        rms_w = tl.load(rms_w_ptrs)

        a = a * rms_w
        b = tl.load(w1_ptrs)

        acc1 += tl.dot(a, b)
        c = tl.load(w3_ptrs)

        acc2 += tl.dot(a, c)

        a_ptrs += BLOCK_SIZE_K * stride_ak
        w1_ptrs += BLOCK_SIZE_K * stride_w1k
        w3_ptrs += BLOCK_SIZE_K * stride_w3k

        rms_w_ptrs += BLOCK_SIZE_K * stride_rms_w

    a_mean = tl.sum(a_sum, axis=1) / K + EPS
    a_norm = tl.math.rsqrt(a_mean)
    acc1 = acc1 * a_norm[:, None]
    acc2 = acc2 * a_norm[:, None]
    accumulator = (acc1 * tl.sigmoid(acc1)) * acc2

    offs_outm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_outn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    out_ptrs = out_ptr + (stride_outm * offs_outm[:, None] + stride_outn * offs_outn[None, :])
    out_mask = (offs_outm[:, None] < M) & (offs_outn[None, :] < N)
    tl.store(out_ptrs, accumulator, mask=out_mask)


def kernel_ff(x: torch.Tensor, w1: torch.Tensor, w3: torch.Tensor, rms_w: torch.Tensor) -> torch.Tensor:
    assert x.dtype == torch.float16
    assert w1.dtype == w3.dtype == rms_w.dtype
    assert w1.dtype in [torch.int8, torch.float16]
    assert w1.shape == w3.shape

    w1_t = w1.t()
    w3_t = w3.t()

    batch, seq_len, dim = x.shape
    M, K = batch * seq_len, dim

    N = w1_t.shape[1]
    assert K == w1_t.shape[0]
    assert w1_t.shape == w3_t.shape
    x_reshape = x.reshape(M, K)
    out = torch.empty((M, N), dtype=x.dtype, device=x.device)
    grid = lambda META: (triton.cdiv(META["M"], META["BLOCK_SIZE_M"]) * triton.cdiv(META["N"], META["BLOCK_SIZE_N"]),)
    print(grid({"BLOCK_SIZE_M": 16, "BLOCK_SIZE_N": 16, "N": N, "M": M}))
    ff_llama[grid](
        x_reshape, w1_t, w3_t, out, rms_w,
        M, N, K,
        *x_reshape.stride(),
        *w1_t.stride(),
        *w3_t.stride(),
        *out.stride(),
        *rms_w.stride(),
        EPS=1e-6,
        BLOCK_SIZE_M=16, BLOCK_SIZE_N=32, BLOCK_SIZE_K=32,
        num_stages=2, num_warps=4
    )
    out = out.view(batch, seq_len, -1)
    return out

In [14]:
i = torch.randn(2, 16,4096).cuda()

st = time.time()
out_torch = ff(i)
print("torch: ", time.time()-st)

st = time.time()
out_triton = kernel_ff(i, ff.feed_forward.w1.weight, ff.feed_forward.w3.weight, ff.ffn_norm.weight)@ff.feed_forward.w2.weight.T
print("torch: ", time.time()-st)

torch:  0.23256564140319824
(1792,)
torch:  0.016289472579956055


In [15]:
out_torch

tensor([[[ 0.0126,  0.0715,  0.0472,  ..., -0.1161,  0.0428,  0.0746],
         [-0.0455,  0.0250, -0.1096,  ..., -0.1021,  0.0262,  0.0693],
         [-0.0825,  0.0545,  0.0403,  ...,  0.0111,  0.0582, -0.0827],
         ...,
         [ 0.0144, -0.1428, -0.1708,  ..., -0.1583,  0.2104, -0.0109],
         [-0.0476, -0.2114,  0.0355,  ..., -0.0542,  0.0668, -0.0678],
         [ 0.1221,  0.1429, -0.3022,  ...,  0.0332,  0.0286, -0.2109]],

        [[ 0.0943, -0.0548, -0.1305,  ...,  0.1589, -0.0806, -0.0659],
         [ 0.0505, -0.0211,  0.0354,  ...,  0.0117, -0.0780,  0.1617],
         [-0.0732, -0.1194, -0.3743,  ...,  0.0207, -0.1059, -0.2253],
         ...,
         [-0.0586,  0.0240,  0.1846,  ..., -0.0885, -0.0387,  0.0046],
         [-0.0112, -0.1118,  0.1091,  ...,  0.1517, -0.0029,  0.1357],
         [-0.0228, -0.0310, -0.0251,  ..., -0.1539,  0.0511,  0.0127]]],
       grad_fn=<UnsafeViewBackward0>)

In [16]:
out_triton

tensor([[[ 0.0125,  0.0716,  0.0472,  ..., -0.1161,  0.0427,  0.0746],
         [-0.0456,  0.0251, -0.1096,  ..., -0.1021,  0.0263,  0.0692],
         [-0.0825,  0.0545,  0.0403,  ...,  0.0111,  0.0583, -0.0827],
         ...,
         [ 0.0144, -0.1428, -0.1708,  ..., -0.1582,  0.2106, -0.0109],
         [-0.0475, -0.2114,  0.0355,  ..., -0.0542,  0.0669, -0.0677],
         [ 0.1221,  0.1428, -0.3025,  ...,  0.0331,  0.0285, -0.2109]],

        [[ 0.0942, -0.0548, -0.1305,  ...,  0.1589, -0.0806, -0.0659],
         [ 0.0504, -0.0211,  0.0353,  ...,  0.0117, -0.0780,  0.1617],
         [-0.0732, -0.1195, -0.3743,  ...,  0.0208, -0.1058, -0.2253],
         ...,
         [-0.0587,  0.0240,  0.1846,  ..., -0.0885, -0.0387,  0.0047],
         [-0.0111, -0.1118,  0.1091,  ...,  0.1519, -0.0029,  0.1359],
         [-0.0227, -0.0309, -0.0251,  ..., -0.1539,  0.0511,  0.0127]]],
       grad_fn=<UnsafeViewBackward0>)